# Imports

In [1]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

# Load Data

In [2]:
vcf = VCF("data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("data/ps3_gwas.phen", sep="\t", header=None)

In [4]:
pcs = pd.read_csv("data/ps3_gwas.eigenvec", sep=r"\s+", header=None)

# Clean and Align Phenotype Data

In [5]:
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# Clean and Align Genotype Data

In [6]:
pc_matrix = pcs.iloc[:, 2:].values

intercept = np.ones(len(pc_matrix))
C = np.column_stack([intercept, pc_matrix])

# GWAS Loop

In [7]:
# Compute y-residuals
covar_coefs, *_ = np.linalg.lstsq(C, y, rcond=None)
y_resid = y - C @ covar_coefs

# Precompute to save computation later
covar_complete = ~np.any(np.isnan(C), axis=1)

In [8]:
# Set up output file
with open("gwas_results.tsv", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    max_snps = 50
    count = 0

    for v in vcf:
        # if count >= max_snps:
        #     break

        # Convert genotype to dosage-like numeric (0/1/2); missing -> nan
        g = v.gt_types.astype(float)
        g[g == 3] = np.nan

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y) & covar_complete
        g2 = g[mask]
        y2 = y_resid[mask]
        C2 = C[mask]

        # Skip if not enough samples
        if len(g2) < C2.shape[1] + 2:
            continue

        # Compute g-residuals
        covar_coefs_g, *_ = np.linalg.lstsq(C2, g2, rcond=None)
        g_resid = g2 - C2 @ covar_coefs_g 

        # MAF on non-missing genotypes
        maf = min(g2.mean() / 2, 1 - g2.mean() / 2)
        if maf < 0.01:
            continue

        # Regression on filtered samples
        denom = g_resid @ g_resid
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g_resid @ y2) / denom

        df = len(y2) - C2.shape[1] - 1
        resid = y2 - beta * g_resid
        se = np.sqrt((resid @ resid) / df / denom)

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        # Check SNPs
        snp_id = v.ID if v.ID is not None else f"{v.CHROM}:{v.POS}"

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{snp_id}\t{beta}\t{p}\t{len(y2)}\n")
        count += 1